# AMMS 302 — Week 14: Techniques when dealing with Big Data
**Chunking · dtype downcast · Parquet · DuckDB views · UPDATE/DELETE · Permission management**

> เปิดคู่กับ [สไลด์ wk14](./wk14.html) — Lab: benchmark chunking vs duckdb, parquet conversion, safe UPDATE pattern

### 🎯 Learning objectives (CLO2/CLO4)
- ประมาณ RAM footprint + เลือกเทคนิค (chunk/downcast/columnar) ให้เหมาะกับขนาดข้อมูล
- ใช้ DuckDB query CSV/Parquet in-place + CREATE VIEW/UPDATE อย่างปลอดภัย
- อธิบาย permission models: file perms (SQLite) / dataset ACL & IAM (BigQuery)

### 📚 Official references
- pandas: [read_csv chunksize](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [to_parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html) · [select_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html)
- Parquet format: [parquet.apache.org](https://parquet.apache.org/)
- DuckDB: [Python guide](https://duckdb.org/docs/stable/clients/python/overview.html) · [CREATE VIEW](https://duckdb.org/docs/stable/sql/statements/create_view.html) · [UPDATE](https://duckdb.org/docs/stable/sql/statements/update.html)
- BigQuery access control: [IAM roles](https://cloud.google.com/bigquery/docs/access-control) · [dataset ACL](https://cloud.google.com/bigquery/docs/dataset-access-controls)

---


In [ ]:
# §0 Setup — สร้าง "big" CSV ~200k rows เพื่อทดสอบ (จำลอง big data บนเครื่องธรรมดา)
import numpy as np, pandas as pd, pathlib, time
rng = np.random.default_rng(7)
big_path = pathlib.Path('big_vitals.csv')
if not big_path.exists():
    n = 200_000
    pd.DataFrame({
      'patient_id': rng.integers(1, 5001, n),
      'visit_no':   rng.integers(1, 40, n),
      'age':        rng.integers(18, 90, n),
      'gender':     rng.choice(['M','F'], n),
      'hba1c':      rng.normal(6.9, 1.4, n).round(2).clip(3,16),
      'sbp':        rng.normal(132, 22, n).round(0),
      'dbp':        rng.normal(84, 12, n).round(0),
      'smoke':      rng.choice([0,1], n, p=[.7,.3]),
    }).to_csv(big_path, index=False)
print(big_path, f"{big_path.stat().st_size/1e6:.1f} MB")

## §1 RAM math — รู้ตัวก่อน OOM (สไลด์ 02)
float64 = 8 bytes/ค่า · 100 cols × 10M rows ≈ 8 GB — โหลดทั้งก้อนเมื่อ RAM 4–8GB → crash!


In [ ]:
# §1 วัด memory จริง
t0=time.time()
df_full = pd.read_csv('big_vitals.csv')
print(f"full load: {time.time()-t0:.2f}s")
mem_mb = df_full.memory_usage(deep=True).sum()/1e6
print(f"memory_usage: {mem_mb:.1f} MB | dtypes:\n", df_full.dtypes.value_counts())
# 💡 ถ้าไฟล์ ×50 (10GB) — เดี๋ยวเครื่องแล็บพังก่อนได้คำตอบ!

## §2 Chunking — อ่านเป็นชุด (สไลด์ 03)
`chunksize=` → iterator ของ DataFrames; aggregate สะสมทีละ chunk แล้ว combine ปลายทาง


In [ ]:
# §2 streaming aggregate: mean hba1c + n per gender โดยไม่ load ทั้งไฟล์
t0 = time.time()
acc = {}   # gender -> [sum_a1c, n]
for chunk in pd.read_csv('big_vitals.csv', chunksize=50_000,
                         usecols=['gender','hba1c']):     # ← usecols ลด IO ด้วย!
    for g, sub in chunk.groupby('gender'):
        s, n = acc.get(g, (0.0, 0))
        acc[g] = (s + sub['hba1c'].sum(), n + len(sub))
out = pd.DataFrame({g:{'mean_hba1c': round(s/n,3), 'n': n} for g,(s,n) in acc.items()}).T
display(out)
print(f"streamed in {time.time()-t0:.2f}s — peak memory ≈ 1 chunk only ✅")

## §3 Downcasting + categories (สไลด์ 04)


In [ ]:
# §3 shrink dtypes
small = df_full.copy()
for c in small.select_dtypes('integer').columns:
    small[c] = pd.to_numeric(small[c], downcast='integer')
for c in ['gender']:
    small[c] = small[c].astype('category')
mem2 = small.memory_usage(deep=True).sum()/1e6
print(f"before {mem_mb:.1f} MB → after {mem2:.1f} MB ({100*(1-mem2/mem_mb):.0f}% saved)")
display(small.dtypes)

## §4 Parquet + DuckDB views (สไลด์ 05–06)
columnar binary format — compression + predicate pushdown; DuckDB อ่าน/เขียน native


In [ ]:
# §4.1 CSV → Parquet (ผ่าน DuckDB COPY)
import duckdb
con = duckdb.connect()
con.execute("COPY (SELECT * FROM read_csv_auto('big_vitals.csv')) TO 'big_vitals.parquet' (FORMAT PARQUET)")
pq = pathlib.Path('big_vitals.parquet')
csv_sz = big_path.stat().st_size/1e6; pq_sz = pq.stat().st_size/1e6
print(f"CSV {csv_sz:.1f} MB → Parquet {pq_sz:.1f} MB ({100*(1-pq_sz/csv_sz):.0f}% smaller)")

# §4.2 VIEW = ตารางเสมือน ชี้ไฟล์ ไม่ copy data
con.execute("CREATE OR REPLACE VIEW v_vitals AS SELECT * FROM read_parquet('big_vitals.parquet')")
t0=time.time()
res = con.execute("""
  SELECT age/10*10 decade, ROUND(AVG(sbp),1) avg_sbp, COUNT(*) n
  FROM v_vitals GROUP BY decade ORDER BY decade""").df()
display(res); print(f"query over view: {time.time()-t0:.2f}s")

## §5 Safe table updates (สไลด์ 07–08)
UPDATE/DELETE บน persistent db — กฎ: **BEGIN → check affected rows → COMMIT / ROLLBACK**


In [ ]:
# §5 ใช้ healthinfo.db ต่อ — UPDATE แบบมี transaction guard
import sqlite3, pathlib
assert pathlib.Path('healthinfo.db').exists(), 'run week05 notebook first'
con_s = sqlite3.connect('healthinfo.db'); cur = con_s.cursor()

before = cur.execute("SELECT COUNT(*) FROM patients WHERE systolic_bp > 220").fetchone()[0]
print("rows to fix (bp>220):", before)

cur.execute("BEGIN")
cur.execute("UPDATE patients SET systolic_bp = NULL WHERE systolic_bp > 220")  # mark implausible
changed = cur.execute("SELECT changes()").fetchone()[0]
if changed == before:
    con_s.commit(); print(f"✅ committed {changed} rows (NULLed implausible bp)")
else:
    con_s.rollback(); print("⚠️ mismatch → rolled back")

# DELETE pattern: ทดสอบก่อนด้วย SELECT เสมอ
preview = cur.execute("SELECT COUNT(*) FROM patients WHERE patient_id NOT IN (SELECT DISTINCT patient_id FROM prescriptions)").fetchone()[0]
print(f"patients w/o rx (DELETE candidates — เราไม่ลบจริง!): {preview}")
con_s.close()

## §6 Permission management (สไลด์ 09)
| ระบบ | Model | คำสั่งหลัก |
|---|---|---|
| SQLite | file-level OS perms + encryption แยก | `chmod 600 healthinfo.db` |
| DuckDB | single-user local (same as file) | — |
| BigQuery | IAM roles per dataset/table | `roles/dataViewer` read-only, `dataEditor` write |

**Principle of least privilege:** analyst ได้ Viewer เท่านั้น · ETL service account ได้ Editor เฉพาะ staging dataset · audit logs เปิดเสมอ

🔗 BigQuery: https://cloud.google.com/bigquery/docs/access-control


### ✅ Self-check
- full-load memory วัดได้ + อธิบาย 8 bytes/value
- chunking ให้ mean เท่า full-load (เทียบได้)
- parquet เล็กกว่า csv ≥60% · view query <1s
- UPDATE rollback-guard ทำงานถูก (changes==before)

### 📝 Homework 14
1) scale test: สร้าง CSV 2× ใหญ่ขึ้น (400k) → เทียบเวลา chunking(20k/chunk) vs duckdb — ส่งตาราง benchmark  
2) เขียน ½ หน้า: ออกแบบ permission plan สำหรับ project กลุ่ม (ใคร role อะไร บน BigQuery) — ส่ง .ipynb

---
### 🔗 Specs
[Parquet](https://parquet.apache.org/) · [DuckDB SQL](https://duckdb.org/docs/stable/sql/introduction.html) · [BQ IAM](https://cloud.google.com/bigquery/docs/access-control)
